In [0]:
# import neceassary spark modules
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

**Silver** **Brands**

In [0]:

# Show bronze brands data  
df_brands = spark.table(f'{catalog_name}.bronze.brz_brands')
df_brands.show(10)

In [0]:
# trim column to remove empty spaces
df_silver = df_brands.withColumn('brand_name', F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
# Remove special charcters
df_silver = df_silver.withColumn('brand_code', F.regexp_replace(F.col('brand_code'), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

In [0]:
# List unique category_code 
df_silver.select('category_code').distinct().show(10)


In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(anomalies, subset='category_code')
df_silver.show(100)

In [0]:
# List unique category_code 
df_silver.select('category_code').distinct().show(10)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f'{catalog_name}.silver.slv_brands')

**Silver Category**

In [0]:
# Show bronze category data
df_category = spark.table(f'{catalog_name}.bronze.brz_category')
df_category.show(100)

In [0]:
# make column 'category_code' capital
df_category = df_category.withColumn('category_code', F.upper(F.col('category_code')))
df_category.show(100)
 

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_category)
df_category.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f'{catalog_name}.silver.slv_category')

**Silver Producs**

In [0]:
# Show bronze products data
df_products = spark.table(f'{catalog_name}.bronze.brz_products')
# df_products.show(100)
display(df_products)

In [0]:
# Make colmuns 'category_code' and 'brand_code' capital
df_products = df_products.withColumn('category_code', F.upper(F.col('category_code')))
df_products = df_products.withColumn('brand_code', F.upper(F.col('brand_code')))
df_products.show(100)


In [0]:
# Convert colmun 'length_cm' to float
df_products = df_products.withColumn('length_cm', F.regexp_replace(F.col('length_cm'), ',', '.'))
df_products = df_products.withColumn('length_cm', F.col('length_cm').cast('float'))
df_products.show(100)

In [0]:
# Convert colmun 'weight_grams' to float
df_products = df_products.withColumn('weight_grams', F.regexp_replace(F.col('weight_grams'), 'g', ''))
df_products = df_products.withColumn('weight_grams', F.col('weight_grams').cast('float'))
df_products.show(100)


In [0]:
# Replace all the NULL value in column 'color' with 'Unknown'
df_products = df_products.withColumn('color', F.when(F.col('color').isNull(), 'Unknown').otherwise(F.col('color')))
df_products.show(100)

In [0]:
display(df_products)

In [0]:
# Limit columns 'length_cm', 'width_cm' and height_cm' to 2 decimal places
df_products = df_products.withColumn('length_cm', F.round(F.col('length_cm'), 2))
df_products = df_products.withColumn('width_cm', F.round(F.col('width_cm'), 2))
df_products = df_products.withColumn('height_cm', F.round(F.col('height_cm'), 2))
df_products.show(100)

In [0]:
display(df_products)

In [0]:
# Make the first letter in column 'color' capital and the rest lower case
df_products = df_products.withColumn('color', F.initcap(F.col('color')))
df_products.show(100)

In [0]:
display(df_products)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_products)
df_products.write.format('delta')\
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable(f'{catalog_name}.silver.slv_products')

**Silver Customer**

In [0]:
# Show bronze customers data
df_customers = spark.table(f'{catalog_name}.bronze.brz_customers')
display(df_customers)

In [0]:
# How many unique 'country_code' in the table?
df_customers = df_customers.select('country_code').distinct()
df_customers.show()


In [0]:
# Get unique 'phone' in the table.
df_customers = spark.table(f'{catalog_name}.bronze.brz_customers')
df_customers.show()

In [0]:
# Remove the '.' in the 'phone' column
df_customers = df_customers.withColumn('phone', F.regexp_replace(F.col('phone'), '\.', ''))
df_customers.show()

In [0]:
# Get unique 'country' in the table.
df_customers_country = df_customers.select('country').distinct()
df_customers_country.show()

In [0]:
# Get unique 'phone' in the table.
df_customers_phone = df_customers.select('phone').distinct()
df_customers_phone.show()

In [0]:
# Format the 'phone' column according to 'country' column and create new column 'phone_2'

from pyspark.sql import functions as F

df_customers = (
    df_customers
    .withColumn("phone_str", F.col("phone").cast("string"))
    .withColumn(
        "phone_2",
        F.when(F.col("phone").isNull(), None)

        # India (+91 followed by 10-digit mobile number)
        .when(
            F.col("country") == "India",
            F.concat(F.lit("+91 "), F.expr("right(phone_str, 10)"))
        )

        # Australia (+61 followed by 9 digits)
        .when(
            F.col("country") == "Australia",
            F.concat(F.lit("+61 "), F.expr("right(phone_str, 9)"))
        )

        # United Kingdom (+44 followed by 10 digits)
        .when(
            F.col("country") == "United Kingdom",
            F.concat(F.lit("+44 "), F.expr("right(phone_str, 10)"))
        )

        # United States (+1 followed by 10 digits)
        .when(
            F.col("country") == "United States",
            F.concat(F.lit("+1 "), F.expr("right(phone_str, 10)"))
        )

        # Canada (+1 followed by 10 digits)
        .when(
            F.col("country") == "Canada",
            F.concat(F.lit("+1 "), F.expr("right(phone_str, 10)"))
        )

        # UAE (+971 followed by 9 digits)
        .when(
            F.col("country") == "United Arab Emirates",
            F.concat(F.lit("+971 "), F.expr("right(phone_str, 9)"))
        )

        # Singapore (+65 followed by 8 digits)
        .when(
            F.col("country") == "Singapore",
            F.concat(F.lit("+65 "), F.expr("right(phone_str, 8)"))
        )

        # Default: retain original value
        .otherwise(F.col("phone_str"))
    )
    .drop("phone_str")
)
display(df2)
# df2.show(truncate=False)

In [0]:
# Replace the NULL values of 'phone' and 'phone_2' with country code of column 'country'
from pyspark.sql import functions as F

df_customers = (
    df_customers
    .withColumn(
        "phone",
        F.when(
            F.col("phone").isNull(),
            F.when(F.col("country") == "India", "91")
             .when(F.col("country") == "Australia", "61")
             .when(F.col("country") == "United Kingdom", "44")
             .when(F.col("country") == "United States", "1")
             .when(F.col("country") == "Canada", "1")
             .when(F.col("country") == "United Arab Emirates", "971")
             .when(F.col("country") == "Singapore", "65")
        ).otherwise(F.col("phone").cast("string"))
    )
    .withColumn(
        "phone_2",
        F.when(
            F.col("phone_2").isNull(),
            F.when(F.col("country") == "India", "+91")
             .when(F.col("country") == "Australia", "+61")
             .when(F.col("country") == "United Kingdom", "+44")
             .when(F.col("country") == "United States", "+1")
             .when(F.col("country") == "Canada", "+1")
             .when(F.col("country") == "United Arab Emirates", "+971")
             .when(F.col("country") == "Singapore", "+65")
        ).otherwise(F.col("phone_2"))
    )
)
display(df_customers)
# df_result.show(truncate=False)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_customers)
df_customers.write.format("delta")\
.mode("overwrite")\
.option("overwriteSchema", "true")\
.saveAsTable(f'{catalog_name}.silver.slv_customers')